# Company Intelligence Agentic System — Demo Notebook

This notebook walks through the two-agent LangGraph system step by step:

1. **Agent 1 (Data Collector)** — pulls real stock data (yfinance) and recent news (DuckDuckGo).
2. **Agent 2 (Analyst)** — computes a real volatility figure and writes a summary, insights, and risk factors.
3. **Orchestrator (LangGraph)** — runs Collector -> Analyst and keeps memory across turns.

Requires a `GROQ_API_KEY` in a `.env` file in this folder (free key: https://console.groq.com/keys).

In [ ]:
from llm import get_llm
from graph import build_graph

llm = get_llm()
app = build_graph(llm)
print("Graph compiled:", app)

## Inspect the individual tools first

Before trusting the agents, let's confirm the raw tools work and see exactly what data they return.

In [ ]:
from tools import get_stock_performance, get_company_news, calculate_volatility

stock_data = get_stock_performance.invoke({"ticker": "NVDA"})
stock_data

In [ ]:
news = get_company_news.invoke({"company": "Nvidia", "max_results": 3})
news

In [ ]:
vol = calculate_volatility.invoke({"prices": stock_data["last_10_closes"]})
vol

## Turn 1: Run the full graph on NVDA

This single call runs Collector -> Analyst automatically. The agents decide for themselves
when and how to call their tools (this is what makes them *agents* rather than a fixed script).

In [ ]:
thread_config = {"configurable": {"thread_id": "notebook-demo"}}

result_1 = app.invoke({"company": "NVDA"}, config=thread_config)

print("=== COLLECTOR REPORT ===")
print(result_1["collector_report"])
print("\n=== ANALYST REPORT ===")
print(result_1["analyst_report"])

## Turn 2: Ask about a second company on the SAME thread

Because we reuse `thread_config` with the same `thread_id`, LangGraph's checkpointer carries
the `session_history` state forward. Watch the Analyst reference NVDA (from turn 1) when
discussing this new company — that's the cross-agent-call memory requirement in action.

In [ ]:
result_2 = app.invoke({"company": "AMD"}, config=thread_config)

print("=== ANALYST REPORT (AMD, with memory of NVDA) ===")
print(result_2["analyst_report"])

In [ ]:
# Inspect the accumulated session memory directly
result_2["session_history"]

## Turn 3: A third company, to confirm memory keeps accumulating


In [ ]:
result_3 = app.invoke({"company": "Tesla"}, config=thread_config)

print("=== ANALYST REPORT (Tesla, with memory of NVDA + AMD) ===")
print(result_3["analyst_report"])
print("\nFull session history now has", len(result_3["session_history"]), "entries")